# 14.3 Q／K 的尺度怎麼影響注意力？


你想在兩個候選位置中找出與當前問題相近的一個。如果它們方向一樣接近，僅有一個向量的數字放大100倍，注意力會不會偏向它？先回顧[查詢與被查詢](https://birdhackor.github.io/tiny-perceptron-vlm/3.3.html)：Query（Q）與Key（K）是同一組特徵轉換出的兩種向量，點積逐項相乘再相加。點積除了反映方向，也會隨向量長度增加，因此高分未必只代表方向更合適。

取 `Q=[1,1]`，兩個K分別是 `[1,0]` 和 `[0,100]`。Q位在對角方向，與水平、垂直方向的夾角都45度；但點積分別為1和100。softmax把分數轉成總和為一的非負權重，用分數的指數再除以總和，於是第二個位置幾乎獨占注意力。這是長度改變造成的偏好，不是我們改了查詢或候選的方向。

可以先將Q、K各自除以自己的長度。例如Q的長度是 `sqrt(1²+1²)=sqrt(2)`，會變成 `[0.7071,0.7071]`；兩個K則變成 `[1,0]` 與 `[0,1]`。這叫L2 normalization，正規化後的點積就是夾角餘弦。兩個分數現在同為0.7071，softmax得到各0.5。這只是QK正規化的一種選擇：其他模型可能採用[均方根尺度](https://birdhackor.github.io/tiny-perceptron-vlm/14.2.html)，應清楚說明實際規則。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
import torch.nn.functional as F

q = torch.tensor([[1.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 100.0]])
raw = q @ k.T
unit = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).T
print("原分數", raw.tolist())
print("原權重", raw.softmax(-1).tolist())
print("正規化分數", unit.round(decimals=4).tolist())
print("正規化權重", unit.softmax(-1).tolist())
print("有方向差的權重", torch.tensor([1.0, 0.0]).mul(4).softmax(-1).tolist())

輸入是一列Q與兩列K，`k.T`把K轉置，讓矩陣乘法一次比較兩個候選。`dim=-1`指定沿最後一軸，也就是各向量的特徵；若沿候選軸正規化，就變成另一個問題。前三種結果應對照上述手算：原分數 `[1,100]`，原權重接近 `[0,1]`，正規化後權重 `[0.5,0.5]`。

最後一行另外示範「控制分數尖銳程度」：假設兩方向分數為1與0，乘4後softmax約 `[0.9820,0.0180]`，比不乘4時的 `[0.7311,0.2689]`更集中。這個乘數常稱scale；若用除數表達，便稱temperature（溫度），溫度越低越集中。正規化拿掉長度差後，仍能另行設定或學習這個尺度，讓模型調整選擇強度。

練習只把K中的100改成0.1，先預測原權重如何變、正規化權重是否變。核對後應看見原分數變成 `[1,0.1]`，而兩個方向相同，正規化仍各0.5。不要把這個例子推成「長度一定無用」；模型可能學會用長度表達訊息，改規則需要重新訓練與比較。
